# 👁️ Pupil-Limbus Segmentation Model Training on Google Colab 🚀

Welcome to the Google Colab training notebook for the Pupil-Limbus eye tracker!  
By using Google Colab's high-performance cloud GPUs (T4, L4, or A100), you can train **up to 20x faster** than on a local CPU.

### 📋 Workflow Overview:
1. Verify GPU is active
2. Clone the project code from GitHub *(no ZIP uploads needed)*
3. Upload your `clinical_data.zip` from Google Drive
4. Install dependencies
5. Verify annotations
6. Run training
7. Export ONNX and download your trained model


In [ ]:
# @title 1. Verify GPU Acceleration 🏎️
import torch
if torch.cuda.is_available():
    d = torch.cuda.get_device_properties(0)
    print(f'✅ GPU active: {torch.cuda.get_device_name(0)}')
    print(f'   Total GPU memory: {d.total_memory / (1024**3):.2f} GB')
    print(f'   CUDA version:     {torch.version.cuda}')
else:
    print('❌ GPU is NOT active!')
    print('   Go to: Runtime -> Change runtime type -> GPU (T4 GPU is free) -> Save')
    print('   Then restart the runtime and run all cells again.')


## 2. Clone Project Code from GitHub 📦

This pulls the **latest committed code** directly from GitHub.  
No ZIP files, no nested folder issues!


In [ ]:
# @title 2. Clone / Update Project from GitHub 🔄
import os, shutil
from pathlib import Path

# CRITICAL: always reset cwd to /content FIRST.
os.chdir('/content')
print(f'Working directory: {os.getcwd()}')

WORKSPACE = Path('/content/workspace')
REPO_URL  = 'https://github.com/Prince649294u83/Centration.git'

if WORKSPACE.exists() and (WORKSPACE / '.git').exists():
    print('🔄 Workspace already cloned - pulling latest changes...')
    !git -C /content/workspace pull
    # Clear Python bytecode cache to force reimport
    for d in WORKSPACE.rglob('__pycache__'):
        shutil.rmtree(d)
        print(f'   Cleared cache: {d}')
else:
    if WORKSPACE.exists():
        shutil.rmtree(WORKSPACE)
    print(f'📥 Cloning from {REPO_URL} ...')
    !git clone https://github.com/Prince649294u83/Centration.git /content/workspace

# Verify critical files exist
print('\n📂 Verifying project structure...')
ok = True
for f in ['scripts/train_model.py', 'scripts/export_onnx.py', 'pupil_tracking']:
    path = WORKSPACE / f
    if path.exists():
        print(f'   ✅ {f}')
    else:
        print(f'   ❌ MISSING: {f}')
        ok = False

if ok:
    print('\n✅ Project code is ready!')
else:
    print('\n❌ Some files are missing. Check the repository URL and try again.')


## 3. Connect Your Clinical Data 📁

Choose **one** option:
- **Option A** *(Recommended)*: Mount Google Drive
- **Option B**: Upload directly to Colab


In [ ]:
# @title Option A: Mount Google Drive & Copy Clinical Data 📁 (Recommended)
from google.colab import drive
import shutil
from pathlib import Path

try:
    drive.mount('/content/drive')
    print('✅ Google Drive mounted!')
    drive_zip = Path('/content/drive/MyDrive/clinical_data.zip')
    local_zip = Path('/content/clinical_data.zip')
    if drive_zip.exists():
        shutil.copy2(drive_zip, local_zip)
        mb = local_zip.stat().st_size / 1024 / 1024
        print(f'   ✅ Copied! ({mb:.1f} MB)')
    else:
        print('   ℹ️  clinical_data.zip not found in Drive root.')
except Exception as e:
    print(f'⚠️  Drive mount failed: {e}')


In [ ]:
# @title Option B: Upload clinical_data.zip Directly 📤
from google.colab import files
print('📤 Select your clinical_data.zip file...')
uploaded = files.upload()
for name, data in uploaded.items():
    dest = f'/content/{name}'
    with open(dest, 'wb') as f:
        f.write(data)
    mb = len(data) / 1024 / 1024
    print(f'   ✅ Uploaded: {dest} ({mb:.1f} MB)')


In [ ]:
# @title Extract Clinical Data into Workspace 🔓
import zipfile, shutil
from pathlib import Path

WORKSPACE = Path('/content/workspace')
data_zip  = Path('/content/clinical_data.zip')
data_dest = WORKSPACE / 'clinical_data'

if not data_zip.exists():
    print('❌ clinical_data.zip not found. Run Option A or B first.')
else:
    if data_dest.exists():
        shutil.rmtree(data_dest)
    mb = data_zip.stat().st_size / 1024 / 1024
    print(f'📦 Extracting ({mb:.1f} MB)...')
    with zipfile.ZipFile(data_zip, 'r') as zf:
        members = zf.namelist()
        roots = set(m.split('/')[0] for m in members if m.split('/')[0])
        if len(roots) == 1 and 'clinical_data' in roots:
            zf.extractall(WORKSPACE)
        else:
            zf.extractall(data_dest)
    # Verify
    ann_file = data_dest / 'annotations' / 'annotations.json'
    img_dir  = data_dest / 'training_data' / 'images'
    if ann_file.exists() and img_dir.exists():
        n = len(list(img_dir.glob('*.jpg'))) + len(list(img_dir.glob('*.png')))
        print(f'✅ Clinical data ready! {n} images found.')
    else:
        print('❌ Extraction issue. Check zip structure.')


## 4. Install Dependencies 🛠️


In [ ]:
# @title Install Core Libraries 🚀
import os, sys
os.chdir('/content/workspace')
sys.path.insert(0, '/content/workspace')

!pip install -q segmentation-models-pytorch albumentations==1.4.3 onnxruntime-gpu timm onnxscript

import torch
print(f'✅ torch {torch.__version__} (CUDA: {torch.cuda.is_available()})')
from pupil_tracking.ml.trainer import Trainer
print('✅ All imports OK!')


## 5. Verify Annotations ✅

**Run this before training!** You should see `with_pupil: 166` (not 0).


In [ ]:
# @title Verify Annotation Parsing 📊
import os, sys
os.chdir('/content/workspace')
sys.path.insert(0, '/content/workspace')

from pupil_tracking.ml.dataset import load_annotations, get_annotation_stats
ids, anns = load_annotations('clinical_data/annotations/annotations.json')
stats = get_annotation_stats(anns)

print(f'Total: {stats["total_images"]}, with pupil: {stats["with_pupil"]}, with limbus: {stats["with_limbus"]}')
if stats['with_pupil'] > 100:
    print('✅ Annotations look correct! Ready to train.')
else:
    print('❌ Re-run step 2 (git pull), then Runtime -> Restart session, then re-run from step 4.')


## 6. Run Training 🏋️

- `--copies-per-image 10` = ~30-40 sec/epoch on T4 (vs ~6 min with default 50)
- 300 epochs ≈ **2-3 hours** total
- After epoch 1 you should see `val_iou > 0.3`


In [ ]:
# @title Start GPU-Accelerated Training 🚀
import os
os.chdir('/content/workspace')

!python scripts/train_model.py --epochs 300 --batch-size 16 --lr 0.0005 --num-classes 3 --copies-per-image 10 --annotation-path clinical_data/annotations/annotations.json --image-dir clinical_data/training_data/images --mask-dir clinical_data/training_data/masks --save-dir models --device cuda


## 7. Export & Download 📤


In [ ]:
# @title Export to ONNX 🌟
import os
from pathlib import Path
os.chdir('/content/workspace')

MODEL = '/content/workspace/models/best_model.pth'
ONNX  = '/content/workspace/models/onnx/segmentation.onnx'

if not Path(MODEL).exists():
    print('❌ Model not found. Complete training first.')
else:
    Path(ONNX).parent.mkdir(parents=True, exist_ok=True)
    !python scripts/export_onnx.py --model "{MODEL}" --output "{ONNX}"
    if Path(ONNX).exists():
        mb = Path(ONNX).stat().st_size / 1024 / 1024
        print(f'✅ ONNX saved: {ONNX} ({mb:.1f} MB)')


In [ ]:
# @title Save to Google Drive 💾
import shutil
from pathlib import Path

DRIVE_DIR  = Path('/content/drive/MyDrive/pupil_limbus_models')
MODELS_DIR = Path('/content/workspace/models')

if not Path('/content/drive').exists():
    print('Drive not mounted. Run Option A first.')
elif not MODELS_DIR.exists():
    print('❌ No models directory.')
else:
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    copied = 0
    for f in MODELS_DIR.rglob('*'):
        if f.is_file() and f.suffix in ('.pth', '.onnx'):
            dest = DRIVE_DIR / f.relative_to(MODELS_DIR)
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dest)
            mb = f.stat().st_size / 1024 / 1024
            print(f'   ✅ {f.name} ({mb:.1f} MB)')
            copied += 1
    print(f'✅ {copied} file(s) saved to Drive.')


In [ ]:
# @title Direct Download ZIP 💾 (Alternative)
import shutil
from google.colab import files
from pathlib import Path

m = Path('/content/workspace/models')
a = '/content/trained_models_colab'
if not m.exists():
    print('❌ No models.')
else:
    shutil.make_archive(a, 'zip', str(m))
    mb = Path(f'{a}.zip').stat().st_size / 1024 / 1024
    print(f'✅ {a}.zip ({mb:.1f} MB)')
    try:
        files.download(f'{a}.zip')
    except Exception as e:
        print(f'Download from sidebar: {a}.zip')
